## Where this notebook is going

- **Run it**: All config is in **agent-backend/.env** (copy from `agent-backend/.env.example`). That one file is used by both the backend and this notebook. Run the env cell below first; never paste secrets in the notebook.
- **Remember different people**: Right now the notebook uses a single `token.json`, so only one Google account per run. To support multiple users (e.g. by Gmail):
  - **Option A – Local profiles**: Use a profile id (e.g. email) and save tokens as `token_{profile}.json`. Each person chooses their profile when they run the notebook; storage is local files only.
  - **Option B – Neon (backend)**: Use the `agent-backend` server and Neon DB. Profiles (and their Google OAuth tokens) are stored in Neon. Users open `/connect/google?profile=alice@gmail.com`, then the notebook or app calls the backend with `profile=alice@gmail.com` for Calendar/Drive. No `client_secrets.json` or token files in the notebook; everything is in the database.
- **Storage**: The backend’s `.env.example` uses `DATABASE_URL` for Postgres; Neon is Postgres. So **yes, profile storage can be on Neon**: the `profiles` table in `agent-backend/server.py` stores one row per profile (e.g. per Gmail) with that user’s OAuth credentials. The notebook can then call the backend’s `/calendar/list` and `/drive/search` with a `profile` parameter instead of doing OAuth itself.

In [1]:
# --- Load config from agent-backend/.env ---
import os
from pathlib import Path

# Notebook reads from that file: GOOGLE_API_KEY_AI (Gemini), API_ENDPOINT (backend URL), BACKEND_API_KEY (header).
# Resolve path so we always load agent-backend/.env (never a different .env from cwd).
try:
    from dotenv import load_dotenv
    cwd = Path.cwd()
    _env = cwd / "agent-backend" / ".env"
    if not _env.exists() and (cwd / "server.py").exists():
        # Running from inside agent-backend: .env is here
        _env = cwd / ".env"
    if not _env.exists():
        _env = cwd.parent / "agent-backend" / ".env"
    if _env.exists():
        load_dotenv(_env, override=True)
    else:
        load_dotenv(override=True)
except ImportError:
    pass

def get_env(key: str, default: str = "") -> str:
    """Read a key from the environment. Never paste secrets in code."""
    return os.environ.get(key, default).strip()

def load_env_keys(required: list[str]) -> dict[str, str]:
    """Load keys from env; return dict. If missing, values are empty and we print a reminder."""
    loaded = {k: get_env(k) for k in required}
    missing = [k for k, v in loaded.items() if not v or v.lower() in ("your_api_key_here", "placeholder", "xxx", "your-gemini-api-key")]
    if missing:
        print("Reminder: set these in agent-backend/.env (see .env.example):", missing)
    return loaded

# Keys used by this notebook: LLM + Option B (backend API). PROFILE_EMAIL is optional (whoever is running).
REQUIRED_KEYS = ["GOOGLE_API_KEY_AI"]
BACKEND_KEYS = ["API_ENDPOINT", "BACKEND_API_KEY"]
env_keys = load_env_keys(REQUIRED_KEYS)
env_keys.update(load_env_keys(BACKEND_KEYS))
env_keys["PROFILE_EMAIL"] = get_env("PROFILE_EMAIL")

Loaded .env from: C:\Users\trinh\Productivity-1-Agentic-AI\agent-backend\.env
GOOGLE_API_KEY_AI from .env starts with: AIza


In [2]:
# --- OAuth-first: no email to type. Run the cell, open the link, sign in; we get your email from Google. ---
import os
import time
import secrets
import requests

API_ENDPOINT = (os.getenv("API_ENDPOINT") or env_keys.get("API_ENDPOINT", "")).rstrip("/")
BACKEND_API_KEY = os.getenv("BACKEND_API_KEY") or env_keys.get("BACKEND_API_KEY", "")
if not API_ENDPOINT or not BACKEND_API_KEY:
    raise RuntimeError("Set API_ENDPOINT and BACKEND_API_KEY in agent-backend/.env (see .env.example).")

def check_profile_connected(profile: str) -> bool:
    r = requests.get(
        f"{API_ENDPOINT}/profiles/check",
        params={"profile": profile},
        headers={"X-Backend-Key": BACKEND_API_KEY},
        timeout=10,
    )
    r.raise_for_status()
    return r.json().get("connected", False)

# OAuth-first: no email to type. Open the link, sign in with Google; we get your email automatically.
state_id = secrets.token_urlsafe(16)
connect_url = f"{API_ENDPOINT}/connect/google?state_id={state_id}"
print("Open this link in a browser and sign in with your Google account (Calendar + Drive):")
print(connect_url)
print("Waiting for you to complete sign-in (we'll detect your email automatically)...")
for _ in range(120):
    time.sleep(2)
    try:
        r = requests.get(f"{API_ENDPOINT}/profiles/by_state", params={"state_id": state_id},
            headers={"X-Backend-Key": BACKEND_API_KEY}, timeout=10)
        if r.status_code == 200:
            profile = r.json().get("profile", "").strip()
            if profile:
                current_profile = profile
                print(f"Using profile: {profile}")
                break
    except Exception:
        pass
else:
    raise RuntimeError("Sign-in did not complete in time. Run the cell again and complete OAuth.")

Open this link in a browser and sign in with your Google account (Calendar + Drive):
http://localhost:8000/connect/google?state_id=g0UvOwtwdMD_ZVq8vppbVA
Waiting for you to complete sign-in (we'll detect your email automatically)...
Using profile: trinhquangminh2910@gmail.com


In [3]:
# Call backend API for Drive search and Calendar list; backend uses tokens stored in Neon for current_profile.
import requests

def _backend_headers():
    return {"X-Backend-Key": BACKEND_API_KEY}

def _search_drive_backend(query: str, profile: str = None):
    p = profile or current_profile
    r = requests.get(
        f"{API_ENDPOINT}/drive/search",
        params={"profile": p, "query": query},
        headers=_backend_headers(),
        timeout=30,
    )
    r.raise_for_status()
    items = r.json()
    if not items:
        return "No files found matching that name."
    return "\n".join([f"File: {it['name']} (ID: {it['id']})" for it in items])

def _list_calendar_events_backend(max_results: int = 10, profile: str = None):
    p = profile or current_profile
    r = requests.get(
        f"{API_ENDPOINT}/calendar/list",
        params={"profile": p, "max_results": max_results},
        headers=_backend_headers(),
        timeout=30,
    )
    r.raise_for_status()
    events = r.json()
    if not events:
        return "No upcoming events found."
    return "\n".join([f"- {e['summary']} at {e['start']}" for e in events])

In [4]:
!pip install langchain
from langchain.tools import tool

@tool
def search_drive(query: str):
    """Search for files in Google Drive by name and return their IDs and names for the current profile."""
    return _search_drive_backend(query, profile=current_profile)

@tool
def list_calendar_events(max_results: int = 10):
    """Fetch upcoming events from the current profile's primary Google Calendar."""
    return _list_calendar_events_backend(max_results=max_results, profile=current_profile)

In [13]:
import os

!pip install langchain-google-genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

# API key from .env (strip quotes if present)
raw = (os.getenv("GOOGLE_API_KEY_AI") or env_keys.get("GOOGLE_API_KEY_AI") or "").strip()
if len(raw) >= 2 and raw[0] == raw[-1] and raw[0] in "\"'":
    raw = raw[1:-1]
api_key = raw
if not api_key or api_key.lower() == "your-gemini-api-key":
    raise RuntimeError("Set GOOGLE_API_KEY_AI in agent-backend/.env (get key from https://aistudio.google.com/apikey).")
if len(api_key) == 64 and all(c in "0123456789abcdef" for c in api_key.lower()):
    raise RuntimeError("GOOGLE_API_KEY_AI looks like STATE_SIGNING_SECRET. Use the Gemini key from aistudio.google.com in agent-backend/.env and re-run the first cell.")
if not api_key.startswith("AIza"):
    raise RuntimeError("GOOGLE_API_KEY_AI should start with 'AIza'. Check agent-backend/.env and re-run the first cell.")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    api_key=api_key
)

all_tools = [search_drive, list_calendar_events]

multi_tool_prompt = """You are a versatile Multi-tool Agent.
1. Use 'search_drive' when the user asks about files, documents, or Drive content.
2. Use 'list_calendar_events' when the user asks about schedule, meetings, events, or calendar.

You MUST call the relevant tool first when the question is about Drive or Calendar. Do not say you lack information until you have called the tool and seen the result.

Important for availability / free time:
- If the user asks when they are FREE, NOT occupied, AVAILABLE, or "when am I not busy": call list_calendar_events (e.g. max_results=30), then infer FREE time from the result. Free time = gaps between events, or times/days that have no events. Report the free slots (e.g. "You're free Monday 2pm–4pm, Tuesday before 10am..."), not the list of busy events.
- If the user asks for "my events" or "what do I have scheduled" or "list my meetings": call list_calendar_events and report the events as listed.
Answer concisely and professionally.
"""

multi_tool_agent = create_agent(
    model=llm,
    tools=all_tools,
    system_prompt=multi_tool_prompt
)

Gemini API key loaded (length 39, starts with AIza)


In [6]:

# Test query 1
test_query_1 = "What files do I have in my drive related to 'BAC'?"

for chunk in multi_tool_agent.stream(
    {"messages": [{"role": "human", "content": test_query_1}]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"Content: {data['messages'][-1].content_blocks}")
        print()

Step: model
Content: [{'type': 'tool_call', 'id': 'b74b3153-b140-4a90-bc3d-d30644cf4e8f', 'name': 'search_drive', 'args': {'query': 'BAC'}}]

Step: tools
Content: [{'type': 'text', 'text': 'File: BAC 2025-2026 Student Life Recognition Credits (ID: 1S6zJp3HdKG5pSv1ReIbjbUkvQTSONF3ac1tLTUl4ZtM)\nFile: BAC 2025-2026 Event Schedule (ID: 1NVKdDu7PLFUHvOJDz3KdtSzMI25Tph-wu_gKLRLTLT4)\nFile: BAC P-VP Application 26-27 (Responses) (ID: 1p2x6OiLxFqsV80P_u6uMvhcor9ATQCNlq3aehdDfvr0)\nFile: BAC P-VP Application 26-27 (ID: 1eGxsPIZYLGJ3DvunKw0jbkYInzGjmpAfNdknXSG_6TY)\nFile: BAC Involvement Fair - Interest Form (ID: 1qMYI4Ko4h7hFHQy1HPg9Mu0B_TKzREmYT1t62dYMuvQ)\nFile: BAC Associate Committee 25-26 Paired Feedback Form (Responses) (ID: 1adbXkmXl8iX5mNI4T1RbEkA0ifpx5NABfYLM677rMDA)\nFile: BAC Associate Committee 25-26 Rotation Schedule (ID: 1wgP1xZFQQNN3JIBIpsbSjEhMsf6Y-aAdi4pyrNlZlYk)\nFile: BAC 2025-2026 Winter Bake Sale Fundraiser Time Slots (ID: 1RWc-fzc9VC7dWAqMzIeju_x7e9P6GEpB0UUwLebiYvg)\nFil

In [10]:
# Test query 2
test_query_2 = "Can you list all events I have next Monday?"

for chunk in multi_tool_agent.stream(
    {"messages": [{"role": "human", "content": test_query_2}]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"Content: {data['messages'][-1].content_blocks}")
        print()

Step: model
Content: [{'type': 'tool_call', 'id': 'e7ee2c64-6d04-455f-8116-fcf247768fed', 'name': 'list_calendar_events', 'args': {'max_results': 20}}]

Step: tools
Content: [{'type': 'text', 'text': "- MATH 318-001 at 2026-03-09T10:00:00-04:00\n- ECON 354-001 at 2026-03-09T12:00:00-04:00\n- MATH 320-001 at 2026-03-09T14:00:00-04:00\n- Mark Stehr's Office Hours at 2026-03-09T16:00:00-04:00\n- BreakThroughTech Weekly Coach Meeting at 2026-03-09T17:00:00-04:00\n- DSAB Weekly E-Board Meeting at 2026-03-09T18:45:00-04:00\n- Creese Student Center Concierge at 2026-03-10T07:00:00-04:00\n- EXAM 081-001 at 2026-03-10T08:00:00-04:00\n- COM 270-901 at 2026-03-10T10:00:00-04:00\n- SORC Office Hours at 2026-03-10T13:30:00-04:00\n- ECON 370-001 at 2026-03-10T14:00:00-04:00\n- BAC Weekly E-Board Meeting at 2026-03-10T18:00:00-04:00\n- Creese Student Center Concierge at 2026-03-10T20:00:00-04:00\n- Winter Q'26 Peer Mentors Introduction at 2026-03-11T09:00:00-04:00\n- DSAB Meeting with the Dean at 202

In [14]:
# Example: availability for the current profile (e.g. "What day is alice@gmail.com available?")
availability_query = f"What days or times is the person with profile {current_profile} are not occupied this week?"
for chunk in multi_tool_agent.stream(
    {"messages": [{"role": "human", "content": availability_query}]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"Content: {data['messages'][-1].content_blocks}")
        print()

Step: model
Content: [{'type': 'tool_call', 'id': '898e0a19-dc7c-4dc1-9e8a-7f1caf651215', 'name': 'list_calendar_events', 'args': {'max_results': 30}}]

Step: tools
Content: [{'type': 'text', 'text': "- MATH 318-001 at 2026-03-09T10:00:00-04:00\n- ECON 354-001 at 2026-03-09T12:00:00-04:00\n- MATH 320-001 at 2026-03-09T14:00:00-04:00\n- Mark Stehr's Office Hours at 2026-03-09T16:00:00-04:00\n- BreakThroughTech Weekly Coach Meeting at 2026-03-09T17:00:00-04:00\n- DSAB Weekly E-Board Meeting at 2026-03-09T18:45:00-04:00\n- Creese Student Center Concierge at 2026-03-10T07:00:00-04:00\n- EXAM 081-001 at 2026-03-10T08:00:00-04:00\n- COM 270-901 at 2026-03-10T10:00:00-04:00\n- SORC Office Hours at 2026-03-10T13:30:00-04:00\n- ECON 370-001 at 2026-03-10T14:00:00-04:00\n- BAC Weekly E-Board Meeting at 2026-03-10T18:00:00-04:00\n- Creese Student Center Concierge at 2026-03-10T20:00:00-04:00\n- Winter Q'26 Peer Mentors Introduction at 2026-03-11T09:00:00-04:00\n- DSAB Meeting with the Dean at 202